# Language Modeling

Language modeling forms the basis of the current AI boom. At it's core, language modeling involves breaking down the structure of language into a representation that machines can understand.

## Tokenization

In machine learning applications, we need a way to parse through sentences, paragraphs, and documents in a way that machines (specifically neural networks) can understand. Consider a fixed dictionary of words. This essentially acts as a massive lookup table, where we can look up words as needed. One problem with using a fixed dictionary of words is that it cannot cope with words not in the dictionary or which are misspelled. It also does not take account of punctuation symbols or other character sequences such as computer code. One alternative is to process words at the character level. However, this appraoch discards the semantically important word structure of language, and the subsequent processing would have to learn to reassemble words from elementary characters.

We can combine the benefits of word and character level representations by converting a string of words and punctuation symbols into a string of **tokens**, which are generally small groups of characters and might
include common words in their entirety, along with fragments of longer words as well as individual characters that can be assembled into less common words.

There are many approaches to tokenization, with the most popular being **byte-pair encoding**. The process starts with the individual characters and iteratively merges them into longer strings. The list of tokens is first initialized with the list of individual characters. Then a body of text is searched for the most frequently occurring adjacent pairs of tokens and these are replaced with a new token. To ensure that words are not merged, a new token is not formed from two tokens if the second token starts with a white space.  

In [ ]:
from collections import defaultdict

def get_pair_counts(word_list):
    """Calculates the frequency of consecutive pairs in a list of words."""
    pair_counts = defaultdict(int)
    for word in word_list:
        for i in range(len(word) - 1):
            pair_counts[(word[i], word[i+1])] += 1
    return pair_counts

def merge_pair(word_list, pair_to_merge, new_token):
    """Merges a given pair into a new token in the word list."""
    merged_word_list = []
    for word in word_list:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair_to_merge:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        merged_word_list.append(new_word)
    return merged_word_list

def train_bpe(corpus, num_merges):
    """Trains a BPE tokenizer on a given corpus."""
    # Initialize vocabulary with individual characters
    vocabulary = set()
    for word in corpus:
        for char in word:
            vocabulary.add(char)
    
    # Pre-tokenize words into lists of characters
    tokenized_corpus = [[char for char in word] for word in corpus]

    merges = {}
    for _ in range(num_merges):
        pair_counts = get_pair_counts(tokenized_corpus)
        if not pair_counts:
            break # No more pairs to merge

        most_frequent_pair = max(pair_counts, key=pair_counts.get)
        new_token = "".join(most_frequent_pair) # Create new subword token
        merges[most_frequent_pair] = new_token
        vocabulary.add(new_token)
        
        tokenized_corpus = merge_pair(tokenized_corpus, most_frequent_pair, new_token)
    
    return merges, vocabulary

def tokenize_with_bpe(text, merges, vocabulary):
    """Tokenizes a given text using learned BPE merges and vocabulary."""
    # Initial tokenization into individual characters
    tokens = [char for char in text]

    # Apply merges in the order they were learned
    for pair, new_token in merges.items():
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    # Handle unknown tokens (if any, though BPE inherently handles this by breaking down)
    final_tokens = []
    for token in tokens:
        if token in vocabulary:
            final_tokens.append(token)
        else:
            # If a token isn't in the final vocabulary (e.g., a single character not merged),
            # it will remain as is or be broken down further if possible.
            # For simplicity, we assume all individual characters are in the initial vocabulary.
            final_tokens.append(token) 
    return final_tokens

# Example Usage:
corpus = ["low", "lower", "lowest", "new", "newer"]
num_merges = 5

merges, vocabulary = train_bpe(corpus, num_merges)
print("Learned Merges:", merges)
print("Final Vocabulary:", vocabulary)

test_text = "newest"
tokenized_text = tokenize_with_bpe(test_text, merges, vocabulary)
print(f"Tokenized '{test_text}': {tokenized_text}")

In [7]:
from transformers import AutoTokenizer

# Load a pretrained BPE tokenizer (GPT-2 uses BPE)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "The quick brown fox jumps over the lazy dog."

# Encode to token IDs
input_ids = tokenizer.encode(text)
print(input_ids)

# Decode back to text
decoded = tokenizer.decode(input_ids)
print(decoded)

[464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]
The quick brown fox jumps over the lazy dog.


## Vector semantics and embeddings

The way that we have traditionally dealt with quantifying relationships between words in a document is to use a co-occurence matrix. Co-ocurrences are simple and intuitive: for a dictionary of words, it counts the frequency that word pairs co-occur. There are, however, many limitations with this approach. Namely, it skews heavily toward common words that have very little importance, such as "the" and "and." There are many alternatives to co-occurence tables such as **TF-IDF** and **pointwise mutual information (PMI)**. We will focus on **word2vec**, which gives a bit of intuition for more advanced techniques such as transformers.

Note that in most cases, word2vec embeddings operate on tokenized word representations. However, for intuition, we will look at the words themselves.

### word2vec
While word2vec is a bit dated for large language modeling at this point, it still offers some crucial insights into how modern vector semantics are represented in terms of embeddings. The representations that we talked about (i.e. co-ocurrence, TD-IDF, and PMI) are long and sparse, whereas word2vec embeddings are short and dense. In other words, a method such as co-ocurrence matrices require that we have a $n \times n$ matrix where $n$ is the number of words that we consider, despite the fact that the majority of (e.g. how often does "aardvark" co-occur with "smoothie"?). In contrast word2vec offers a way to embed the vocabulary information in vectors that retain the original information without getting unwieldy. it turns out that embeddings are much easier to work with for NLP, likely because the weights in a model are easier to learn. Additionally, word2vec embeddings are **static embeddings**, meaning that the method learns one fixed embedding for each word in the vocabulary. 

The main intuition behind word2vec is to train a classifier on a binary prediction task and then take the learned classifier weights as the embeddings. Consider a target word $w$ and context words $c_i$:
$$[c_1, c_2, c_3, w, c_4, c_5]$$
The goal here is to predict appropriateness of the context words given center target word. This paradigm is known as **skip gram** (in contrast to **CBOW**, which predicts the center word from context words.). The probability that $c_i$ is a real context word is $\mathbb{P}(+ |w, c)$ and that probability that it isn't is $\mathbb{P}(- |w, c) = 1-\mathbb{P}(+ |w, c_i)$. To compute the similarity between dense embeddings, we rely on the dot product:
$$\text{Similarity(c, w)} \approx \langle c, w\rangle$$
which can then be converted to a probability via the sigmoid function $\sigma(\cdot)$. Making the assumption that all context words are independent, we have that
$$\mathbb{P}(+ |w, c) = \prod^5_{i=1}\sigma\left(\langle c_i, w \rangle\right) = \frac{1}{1+\exp(-\langle c_i, w \rangle)}$$
This allows up to set up the classification problem where the learnable weights are the target words $W\in \mathbb{R}^{d \times |V|}$ and the context words are $C\in\mathbb{R}^{d \times |V|}$. Here, the context and the target word dictionary are the same size, so we often just use one.

To actually learn the embeddings, we employ a technique called **skip-gram negative sampling (SGNS)**. In order to perform binary classification, we also need negative examples or **noise words**. So, for each $(w, c_{pos})$, we create $k$ negative samples $(w, c_{neg})$. The negative samples are typically chosen according to a **weighted unigram frequency** $\mathbb{P}_\alpha(w)$:
$$\mathbb{P}_\alpha(x) = \frac{\text{count}(x)^\alpha}{\sum_{x'\in V}\text{count}(x')}$$
It was actually shown that SGNS implicitly factorizs a shifted PMI matrix. This explains why it captures semantic similarity so well and connects word2vec mathematically to classical co-occurrence statistics. Once we have the training examples, we want to maximize the similarity of target word, context word pairs drawn from positive examples as well as minimize the similarity of target word, context word pairs drawn from negative examples. We can formulate an appropriate loss function and obtain the learn weights via a very shallow neural network — essentially a single linear layer — where the learned embedding matrix doubles as the model’s hidden weights

While word2vec isn't used too much for modern NLP applications, it does illustrate an important concept that the information and dynamics of words can be captured in the weights of a modell. Specifically, word2vec was the first method to show that simple gradient-based learning of dense embeddings yields meaningful linear semantic structure (e.g., king – man + woman ≈ queen). This is a foundation for modern transformer architectures.

In [13]:
import numpy as np
import pandas as pd
import re 

dataset = pd.read_csv("../../data/Corona_NLP_train.csv", encoding='latin1')

texts=[]
for i in range(0,len(dataset)):
  text = re.sub('[^a-zA-Z]' , ' ', dataset['OriginalTweet'][i])
  text = text.lower()
  text = text.split()
  x = len(text) if text.count('https') ==0  else text.index('https') 
  text = text[: x ]
  text = [t for t in text if not t=='https']
  text = ' '.join(text)
  texts.append(text)

print(len(texts))

41157


In [19]:
from gensim.models import Word2Vec

sentences = [line.split() for line in texts]
w2v = Word2Vec(sentences, vector_size=100, window=5, workers=4, min_count=5)

print(sentences[20:25])

# Find embedding of specific word
print("Embedding of 'chicken': ", w2v.wv["chicken"])
print()

# similarity between words
print("Similarity between 'chicken' and 'flu': ", w2v.wv.similarity('chicken', 'flu'))
print()

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


[['with', 'nations', 'inficted', 'with', 'covid', 'the', 'world', 'must', 'not', 'play', 'fair', 'with', 'china', 'goverments', 'must', 'demand', 'china', 'adopts', 'new', 'guilde', 'lines', 'on', 'food', 'safty', 'the', 'chinese', 'goverment', 'is', 'guilty', 'of', 'being', 'irosponcible', 'with', 'life', 'on', 'a', 'global', 'scale'], [], ['we', 'have', 'amazing', 'cheap', 'deals', 'for', 'the', 'covid', 'going', 'on', 'to', 'help', 'you', 'trials', 'monthly', 'yearly', 'and', 'resonable', 'prices', 'subscriptions', 'just', 'dm', 'us', 'bestiptv', 'iptv', 'service', 'iptv', 'iptvdeals', 'cheap', 'iptv', 'football', 'hd', 'movies', 'adult', 'cinema', 'hotmovies', 'iptvnew', 'iptv', 'adult'], ['we', 'have', 'amazing', 'cheap', 'deals', 'for', 'the', 'covid', 'going', 'on', 'to', 'help', 'you', 'trials', 'monthly', 'yearly', 'and', 'resonable', 'prices', 'subscriptions', 'just', 'dm', 'us', 'bestiptv', 'iptv', 'service', 'iptv', 'iptvdeals', 'cheap', 'iptv', 'football', 'hd', 'movies', 

## Joint distribution of words in sequences

Now that we have a representation of semantic meaning of words, we want to be able to understand the meaning of sequences of words or tokens. In other words, we want to ask the question "how likely is a particular sequence of words?" Mathematically, we ask what is the probability
$$\mathbb{P}(x_1, x_2, \dots, x_n)$$
given a sequence of words $x_1,\dots, x_n$.

### Bag of words (BoW)
For bag of words, we assume words are drawn independently from the same distribution and hence that the joint distribution is fully factorized in the form
$$\mathbb{P}(x_1, x_2, \dots, x_n) = \prod^n_{i=1}\mathbb{P}(x_i)$$ 
There are some obvious major flaws with this approach. The most glaring one is that there is no concept of order (e.g. "that was good, not bad at all" and "that was bad, not good at all" outputs the same probability). Thus, we get the very self-aware naming of "bag of words."

### Autoregressive models
To address the shortcomings of BoW, we can take an autoregressive approach, where each word is conditioned on the previous words in the string. WLOG, we can decompose the distribution over the sequence of words into a product of conditional distributions in the form
$$ \mathbb{P}(x_1, x_2, \dots, x_n) = \prod^n_{i=1}\mathbb{P}(x_i | x_{-i})$$
We can simplify the model dramatically by assuming that each of the conditional distributions on the right-hand side of is independent of all previous observations except the Lmost recent words. For example, if $L=2$, then we have
$$ \mathbb{P}(x_1, x_2, \dots, x_n) = \mathbb{P}(x_1)\mathbb{P}(x_2 | x_1)\prod^n_{i=3}\mathbb{P}(x_i | x_{i-1}, x_{i-2})$$

In either case, the model allows us to come up with probabilities of new words, and therefore generate novel text. Consider the autoregressive model with $L=2$. If we provide the first and second words in a sequence,
then we can sample from the tri-gram statistics $\mathbb{P}(x_i | x_{i-1}, x_{i-2})$ to generate the third word, and then we can use the second and third words to sample the fourth word, and so on. It should be noted that this would likely produce nonesense as real-world text likely has to consider long-range context depdendencies beyond the previous two words. Nevertheless, the autoregressive representation will play a central role when we consider modern language models based not on probability tables but on deep neural networks configured as transformers.

## Summary

This scratches the surface of natural language modeling. However, it does highlight the essential pre-training steps/intuition required for transformer architectures, which undergird modern-day LLMs. We saw how text data can be parsed into machine-level language via tokenization. We then saw how the tokenized representation can be transformed into dense embeddings, capturing the semantic relationships between words. Finally, we see how the sequence of embeddings can be marshalled to allow for generative process that creates new words.